In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

/home/gonzalo/anaconda3/envs/sticker_sales/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [3]:
train["date"] = pd.to_datetime(train["date"])
test["date"] = pd.to_datetime(test["date"])

In [4]:
# categorical_combinations = train.groupby(["country", "store", "product"]).size().sort_values().reset_index()
categorical_columns = ["country", "store", "product"]
categorical_combinations = train[categorical_columns].drop_duplicates()
categorical_combinations["id_column"] = categorical_combinations.index.astype(str)

In [5]:
train_autogluon = train.copy(deep=True)
train_autogluon = train_autogluon.merge(categorical_combinations, on = categorical_columns)
train_autogluon = train_autogluon.drop(columns = categorical_columns + ['id'])

In [6]:
test_autogluon = test.copy(deep=True)
test_autogluon = test_autogluon.merge(categorical_combinations, on = categorical_columns)
test_autogluon = test_autogluon.drop(columns = categorical_columns + ['id'])

In [7]:
prediction_horizon = test_autogluon[test_autogluon["id_column"] == "0"]
prediction_horizon = (prediction_horizon["date"].max() - prediction_horizon["date"].min()).days + 1

In [8]:
train_autogluon_df = TimeSeriesDataFrame(
    train_autogluon,
    id_column = "id_column",
    timestamp_column = "date"
)

predictor = TimeSeriesPredictor(
    prediction_length = prediction_horizon,
    target = "num_sold",
    eval_metric = "mape"
)

In [9]:
predictor.fit(
    train_autogluon_df,
    presets = "medium_quality"
)

Beginning AutoGluon training...
AutoGluon will save models to '/home/gonzalo/Documents/Competitions/Forecasting Sticker Sales/AutogluonModels/ag-20250129_220521'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.16
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #52-Ubuntu SMP PREEMPT_DYNAMIC Thu Dec  5 13:09:44 UTC 2024
CPU Count:          16
GPU Count:          1
Memory Avail:       10.89 GB / 14.90 GB (73.1%)
Disk Space Avail:   252.94 GB / 372.96 GB (67.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAPE,
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 1095,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'num_sold',
 'verbosity': 2}

Inferred time series freque

In [10]:
predictor.leaderboard()

,model,score_val,pred_time_val,fit_time_marginal,fit_order
0,WeightedEnsemble,-0.157630,0.343102,7.190995,8
1,DirectTabular,-0.157630,0.343102,1.218983,4
2,Chronos[bolt_small],-0.267996,4.757210,2.170332,7
3,Naive,-0.282107,1.391465,0.108858,1
4,RecursiveTabular,-0.360025,4.452694,0.920721,3
5,ETS,-0.386305,5.397752,0.118444,5
6,SeasonalNaive,-0.387085,0.105118,0.088732,2
7,Theta,-0.481764,33.584983,0.112435,6


In [11]:
predictions = predictor.predict(train_autogluon_df)
predictions = predictions.reset_index()[["item_id", "timestamp", "mean"]]\
    .rename(columns={"item_id": "id_column", "mean": "num_sold", "timestamp": "date"})
predictions = predictions.merge(categorical_combinations, on='id_column', how='left')
predictions = predictions.merge(test, on=categorical_columns + ["date"])[["id", "num_sold"]]

Model not specified in predict, will default to the model with the best validation score: DirectTabular


In [ ]:
leaderboards_total = []
predictions_total = []
for id in train_autogluon["id_column"].unique():
    if id == "0":
        continue
    train_autogluon_subset = train_autogluon[train_autogluon["id_column"] == f"{id}"]
    train_autogluon_subset_df = TimeSeriesDataFrame(train_autogluon_subset,
                                                    id_column = "id_column",
                                                    timestamp_column = "date")
    predictor = TimeSeriesPredictor(prediction_length = prediction_horizon,
                                    target = "num_sold",
                                    eval_metric = "mape")
    predictor.fit(
    train_autogluon_subset_df,
    presets = "medium_quality")

    leaderboard = predictor.leaderboard()
    leaderboard["id"] = id
    leaderboards_total.append(leaderboard)

    predictions = predictor.predict(train_autogluon_subset_df)
    predictions = predictions.reset_index()[["item_id", "timestamp", "mean"]]\
        .rename(columns={"item_id": "id_column", "mean": "num_sold", "timestamp": "date"})
    predictions = predictions.merge(categorical_combinations, on='id_column', how='left')
    predictions = predictions.merge(test, on=categorical_columns + ["date"])[["id", "num_sold"]]
    predictions_total.append(predictions)

Beginning AutoGluon training...
AutoGluon will save models to '/home/gonzalo/Documents/Competitions/Forecasting Sticker Sales/AutogluonModels/ag-20250129_221302'
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.10.16
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #52-Ubuntu SMP PREEMPT_DYNAMIC Thu Dec  5 13:09:44 UTC 2024
CPU Count:          16
GPU Count:          1
Memory Avail:       9.38 GB / 14.90 GB (62.9%)
Disk Space Avail:   252.80 GB / 372.96 GB (67.8%)
Setting presets to: medium_quality

Fitting with arguments:
{'enable_ensemble': True,
 'eval_metric': MAPE,
 'hyperparameters': 'light',
 'known_covariates_names': [],
 'num_val_windows': 1,
 'prediction_length': 1095,
 'quantile_levels': [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9],
 'random_seed': 123,
 'refit_every_n_windows': 1,
 'refit_full': False,
 'skip_model_selection': False,
 'target': 'num_sold',
 'verbosity': 2}

Inferred time series frequen